# Transformer Architecture — PyTorch Implementation

A ground-up implementation of the original **"Attention Is All You Need"** (Vaswani et al., 2017) Transformer, built purely in PyTorch without using `nn.Transformer`.

**Sections**
1. Imports & device setup
2. Scaled Dot-Product Attention
3. Multi-Head Attention
4. Position-wise Feed-Forward Network
5. Sinusoidal Positional Encoding
6. Encoder Layer + Encoder Stack
7. Decoder Layer + Decoder Stack
8. Full Transformer (Encoder-Decoder)
9. Encoder-Only variant (BERT-style)
10. Decoder-Only variant (GPT-style)
11. Parameter count & forward pass smoke-test

In [1]:
import math
import copy

import torch
import torch.nn as nn
import torch.nn.functional as F

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {DEVICE}")

Using device: mps


---
## 1. Scaled Dot-Product Attention

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right) V$$

- `mask` is an **additive** mask: positions set to `-inf` are excluded after softmax.
- Used for both **causal** (decoder self-attention) and **padding** masks.

In [2]:
def scaled_dot_product_attention(
    q: torch.Tensor,   # (batch, heads, seq_q, d_k)
    k: torch.Tensor,   # (batch, heads, seq_k, d_k)
    v: torch.Tensor,   # (batch, heads, seq_k, d_v)
    mask: torch.Tensor | None = None,
    dropout: nn.Dropout | None = None,
) -> tuple[torch.Tensor, torch.Tensor]:
    d_k = q.size(-1)

    # (batch, heads, seq_q, seq_k)
    scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(d_k)

    if mask is not None:
        scores = scores.masked_fill(mask == 0, float("-inf"))

    attn_weights = F.softmax(scores, dim=-1)

    # Replace NaN rows (full -inf → softmax gives NaN) with 0
    attn_weights = torch.nan_to_num(attn_weights, nan=0.0)

    if dropout is not None:
        attn_weights = dropout(attn_weights)

    # (batch, heads, seq_q, d_v)
    output = torch.matmul(attn_weights, v)
    return output, attn_weights

---
## 2. Multi-Head Attention

Projects Q, K, V into `h` subspaces of dimension `d_k = d_model // h`, runs attention in each, then concatenates and projects back.

$$\text{MultiHead}(Q,K,V) = \text{Concat}(\text{head}_1,\ldots,\text{head}_h)\,W^O$$
$$\text{head}_i = \text{Attention}(QW_i^Q,\; KW_i^K,\; VW_i^V)$$

In [3]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int, dropout: float = 0.1):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        # Single fused projection for efficiency; split after
        self.w_q = nn.Linear(d_model, d_model, bias=False)
        self.w_k = nn.Linear(d_model, d_model, bias=False)
        self.w_v = nn.Linear(d_model, d_model, bias=False)
        self.w_o = nn.Linear(d_model, d_model, bias=False)

        self.dropout = nn.Dropout(dropout)
        self.attn_weights = None  # stored for inspection

    def _split_heads(self, x: torch.Tensor) -> torch.Tensor:
        # x: (batch, seq, d_model) → (batch, heads, seq, d_k)
        batch, seq, _ = x.size()
        return x.view(batch, seq, self.num_heads, self.d_k).transpose(1, 2)

    def _merge_heads(self, x: torch.Tensor) -> torch.Tensor:
        # x: (batch, heads, seq, d_k) → (batch, seq, d_model)
        batch, _, seq, _ = x.size()
        return x.transpose(1, 2).contiguous().view(batch, seq, self.d_model)

    def forward(
        self,
        query: torch.Tensor,
        key: torch.Tensor,
        value: torch.Tensor,
        mask: torch.Tensor | None = None,
    ) -> torch.Tensor:
        q = self._split_heads(self.w_q(query))
        k = self._split_heads(self.w_k(key))
        v = self._split_heads(self.w_v(value))

        attn_out, self.attn_weights = scaled_dot_product_attention(
            q, k, v, mask=mask, dropout=self.dropout
        )

        merged = self._merge_heads(attn_out)   # (batch, seq, d_model)
        return self.w_o(merged)

---
## 3. Position-wise Feed-Forward Network

Applied independently to each position:

$$\text{FFN}(x) = \max(0,\; xW_1 + b_1)\,W_2 + b_2$$

The inner dimension `d_ff` is typically `4 × d_model`.

In [4]:
class PositionwiseFeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

---
## 4. Sinusoidal Positional Encoding

Injects position information without learned parameters:

$$PE_{(pos,\,2i)} = \sin\!\left(\frac{pos}{10000^{2i/d_{\text{model}}}}\right), \quad
PE_{(pos,\,2i+1)} = \cos\!\left(\frac{pos}{10000^{2i/d_{\text{model}}}}\right)$$

In [5]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 5000, dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

        pe = torch.zeros(max_len, d_model)                          # (max_len, d_model)
        position = torch.arange(max_len).unsqueeze(1).float()       # (max_len, 1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )                                                            # (d_model/2,)
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        # Register as buffer: moves with .to(device) but not a learned parameter
        self.register_buffer("pe", pe.unsqueeze(0))                 # (1, max_len, d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (batch, seq, d_model)
        x = x + self.pe[:, : x.size(1)]
        return self.dropout(x)

---
## 5. Encoder Layer

Each encoder layer contains:
1. Multi-head **self-attention** (all positions attend to all)
2. Position-wise feed-forward network

Both sublayers use **residual connections** and **Post-LayerNorm** (original paper convention).

In [6]:
class EncoderLayer(nn.Module):
    def __init__(self, d_model: int, num_heads: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.ffn = PositionwiseFeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(
        self, x: torch.Tensor, src_mask: torch.Tensor | None = None
    ) -> torch.Tensor:
        # Sublayer 1: self-attention + residual + norm
        attn_out = self.self_attn(x, x, x, mask=src_mask)
        x = self.norm1(x + self.dropout(attn_out))

        # Sublayer 2: FFN + residual + norm
        ffn_out = self.ffn(x)
        x = self.norm2(x + self.dropout(ffn_out))
        return x


class Encoder(nn.Module):
    def __init__(self, layer: EncoderLayer, num_layers: int):
        super().__init__()
        # Deep-copy so each layer has independent weights
        self.layers = nn.ModuleList([copy.deepcopy(layer) for _ in range(num_layers)])
        self.norm = nn.LayerNorm(layer.self_attn.d_model)

    def forward(
        self, x: torch.Tensor, src_mask: torch.Tensor | None = None
    ) -> torch.Tensor:
        for layer in self.layers:
            x = layer(x, src_mask)
        return self.norm(x)

---
## 6. Decoder Layer

Each decoder layer contains **three** sublayers:
1. **Masked** multi-head self-attention — causal mask prevents attending to future tokens
2. **Cross-attention** — queries come from the decoder, keys/values from the encoder output
3. Position-wise feed-forward network

In [7]:
class DecoderLayer(nn.Module):
    def __init__(self, d_model: int, num_heads: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.self_attn  = MultiHeadAttention(d_model, num_heads, dropout)  # masked
        self.cross_attn = MultiHeadAttention(d_model, num_heads, dropout)  # encoder-decoder
        self.ffn = PositionwiseFeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(
        self,
        x: torch.Tensor,
        memory: torch.Tensor,
        tgt_mask: torch.Tensor | None = None,
        memory_mask: torch.Tensor | None = None,
    ) -> torch.Tensor:
        # Sublayer 1: causal self-attention
        attn1 = self.self_attn(x, x, x, mask=tgt_mask)
        x = self.norm1(x + self.dropout(attn1))

        # Sublayer 2: cross-attention — Q from decoder, K/V from encoder memory
        attn2 = self.cross_attn(x, memory, memory, mask=memory_mask)
        x = self.norm2(x + self.dropout(attn2))

        # Sublayer 3: feed-forward
        ffn_out = self.ffn(x)
        x = self.norm3(x + self.dropout(ffn_out))
        return x


class Decoder(nn.Module):
    def __init__(self, layer: DecoderLayer, num_layers: int):
        super().__init__()
        self.layers = nn.ModuleList([copy.deepcopy(layer) for _ in range(num_layers)])
        self.norm = nn.LayerNorm(layer.self_attn.d_model)

    def forward(
        self,
        x: torch.Tensor,
        memory: torch.Tensor,
        tgt_mask: torch.Tensor | None = None,
        memory_mask: torch.Tensor | None = None,
    ) -> torch.Tensor:
        for layer in self.layers:
            x = layer(x, memory, tgt_mask, memory_mask)
        return self.norm(x)

---
## 7. Mask Utilities

Two mask types:
- **Padding mask**: hides `<PAD>` tokens from attention
- **Causal (subsequent) mask**: lower-triangular mask for autoregressive decoding

In [8]:
def make_padding_mask(seq: torch.Tensor, pad_idx: int = 0) -> torch.Tensor:
    """
    seq: (batch, seq_len) of token ids
    Returns: (batch, 1, 1, seq_len) bool tensor — 1 where token is real, 0 at pad positions.
    """
    return (seq != pad_idx).unsqueeze(1).unsqueeze(2)


def make_causal_mask(seq_len: int, device: torch.device) -> torch.Tensor:
    """
    Returns: (1, 1, seq_len, seq_len) lower-triangular bool mask.
    Position i can attend to positions 0..i only.
    """
    return torch.tril(torch.ones(seq_len, seq_len, device=device)).bool().unsqueeze(0).unsqueeze(0)


def make_tgt_mask(tgt: torch.Tensor, pad_idx: int = 0) -> torch.Tensor:
    """Combines causal + padding mask for decoder self-attention."""
    seq_len = tgt.size(1)
    pad_mask    = make_padding_mask(tgt, pad_idx)               # (batch, 1, 1, seq_len) bool
    causal_mask = make_causal_mask(seq_len, tgt.device)         # (1, 1, seq_len, seq_len) bool
    return pad_mask & causal_mask                               # (batch, 1, seq_len, seq_len)

---
## 8. Full Encoder-Decoder Transformer

The complete sequence-to-sequence model (e.g., for machine translation):

```
src tokens → Embedding + PE → Encoder → memory
tgt tokens → Embedding + PE → Decoder(memory) → Linear → logits
```

In [9]:
class Transformer(nn.Module):
    def __init__(
        self,
        src_vocab_size: int,
        tgt_vocab_size: int,
        d_model: int = 512,
        num_heads: int = 8,
        num_encoder_layers: int = 6,
        num_decoder_layers: int = 6,
        d_ff: int = 2048,
        max_seq_len: int = 5000,
        dropout: float = 0.1,
        pad_idx: int = 0,
    ):
        super().__init__()
        self.pad_idx = pad_idx
        self.d_model = d_model

        self.src_embedding = nn.Embedding(src_vocab_size, d_model, padding_idx=pad_idx)
        self.tgt_embedding = nn.Embedding(tgt_vocab_size, d_model, padding_idx=pad_idx)
        self.pos_encoding  = PositionalEncoding(d_model, max_seq_len, dropout)

        enc_layer = EncoderLayer(d_model, num_heads, d_ff, dropout)
        dec_layer = DecoderLayer(d_model, num_heads, d_ff, dropout)
        self.encoder = Encoder(enc_layer, num_encoder_layers)
        self.decoder = Decoder(dec_layer, num_decoder_layers)

        self.output_projection = nn.Linear(d_model, tgt_vocab_size)

        self._init_weights()

    def _init_weights(self):
        # Xavier uniform — keeps variance stable through deep stacks
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def encode(self, src: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        src_mask = make_padding_mask(src, self.pad_idx)
        src_emb  = self.pos_encoding(self.src_embedding(src) * math.sqrt(self.d_model))
        memory   = self.encoder(src_emb, src_mask)
        return memory, src_mask

    def decode(
        self,
        tgt: torch.Tensor,
        memory: torch.Tensor,
        memory_mask: torch.Tensor | None = None,
    ) -> torch.Tensor:
        tgt_mask = make_tgt_mask(tgt, self.pad_idx)
        tgt_emb  = self.pos_encoding(self.tgt_embedding(tgt) * math.sqrt(self.d_model))
        return self.decoder(tgt_emb, memory, tgt_mask, memory_mask)

    def forward(self, src: torch.Tensor, tgt: torch.Tensor) -> torch.Tensor:
        memory, src_mask = self.encode(src)
        dec_out = self.decode(tgt, memory, src_mask)
        return self.output_projection(dec_out)  # (batch, tgt_seq, tgt_vocab_size)

---
## 9. Encoder-Only Variant (BERT-style)

Used for classification, NER, extractive QA — tasks that require **bidirectional** context.
- No decoder, no causal mask
- Output: one contextual vector per input token
- A task-specific head (e.g., classification) is attached on top of the `[CLS]` token

In [10]:
class BERTStyleEncoder(nn.Module):
    """Encoder-only Transformer for sequence classification."""

    def __init__(
        self,
        vocab_size: int,
        num_classes: int,
        d_model: int = 768,
        num_heads: int = 12,
        num_layers: int = 12,
        d_ff: int = 3072,
        max_seq_len: int = 512,
        dropout: float = 0.1,
        pad_idx: int = 0,
    ):
        super().__init__()
        self.pad_idx = pad_idx
        self.d_model = d_model

        self.embedding   = nn.Embedding(vocab_size, d_model, padding_idx=pad_idx)
        self.pos_encoding = PositionalEncoding(d_model, max_seq_len, dropout)

        enc_layer    = EncoderLayer(d_model, num_heads, d_ff, dropout)
        self.encoder = Encoder(enc_layer, num_layers)

        # Classification head applied to the [CLS] token (position 0)
        self.classifier = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.Tanh(),
            nn.Dropout(dropout),
            nn.Linear(d_model, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (batch, seq_len) token ids — position 0 is [CLS]
        mask   = make_padding_mask(x, self.pad_idx)
        x_emb  = self.pos_encoding(self.embedding(x) * math.sqrt(self.d_model))
        encoded = self.encoder(x_emb, mask)          # (batch, seq_len, d_model)
        cls_vec = encoded[:, 0, :]                   # (batch, d_model)  — [CLS] token
        return self.classifier(cls_vec)              # (batch, num_classes)

---
## 10. Decoder-Only Variant (GPT-style)

Used for text generation, in-context learning, agentic tasks.
- No encoder, no cross-attention
- Causal mask enforces left-to-right autoregressive generation
- Output: next-token probability distribution over the vocabulary

In [11]:
class CausalDecoderLayer(nn.Module):
    """Decoder-only layer — no cross-attention block."""

    def __init__(self, d_model: int, num_heads: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.ffn       = PositionwiseFeedForward(d_model, d_ff, dropout)
        self.norm1     = nn.LayerNorm(d_model)
        self.norm2     = nn.LayerNorm(d_model)
        self.dropout   = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, causal_mask: torch.Tensor) -> torch.Tensor:
        attn_out = self.self_attn(x, x, x, mask=causal_mask)
        x = self.norm1(x + self.dropout(attn_out))
        ffn_out = self.ffn(x)
        x = self.norm2(x + self.dropout(ffn_out))
        return x


class GPTStyleDecoder(nn.Module):
    """Decoder-only Transformer for causal language modelling."""

    def __init__(
        self,
        vocab_size: int,
        d_model: int = 768,
        num_heads: int = 12,
        num_layers: int = 12,
        d_ff: int = 3072,
        max_seq_len: int = 1024,
        dropout: float = 0.1,
    ):
        super().__init__()
        self.d_model = d_model

        self.embedding    = nn.Embedding(vocab_size, d_model)
        self.pos_encoding = PositionalEncoding(d_model, max_seq_len, dropout)
        self.layers       = nn.ModuleList(
            [CausalDecoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)]
        )
        self.norm   = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

        # Weight tying: embedding and output projection share weights
        self.lm_head.weight = self.embedding.weight

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (batch, seq_len) token ids
        seq_len = x.size(1)
        causal_mask = make_causal_mask(seq_len, x.device)   # (1, 1, seq_len, seq_len)

        x = self.pos_encoding(self.embedding(x) * math.sqrt(self.d_model))
        for layer in self.layers:
            x = layer(x, causal_mask)
        x = self.norm(x)
        return self.lm_head(x)   # (batch, seq_len, vocab_size) — logits

---
## 11. Smoke Tests & Parameter Counts

Verify shapes and count trainable parameters for each model variant.

In [12]:
def count_parameters(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


# ── Encoder-Decoder Transformer (paper default: 65M params) ──────────────────
seq2seq = Transformer(
    src_vocab_size=32_000,
    tgt_vocab_size=32_000,
    d_model=512,
    num_heads=8,
    num_encoder_layers=6,
    num_decoder_layers=6,
    d_ff=2048,
    dropout=0.1,
).to(DEVICE)

batch, src_len, tgt_len = 4, 20, 18
src = torch.randint(1, 32_000, (batch, src_len)).to(DEVICE)
tgt = torch.randint(1, 32_000, (batch, tgt_len)).to(DEVICE)

logits = seq2seq(src, tgt)
print(f"[Encoder-Decoder]  output shape : {logits.shape}")
print(f"[Encoder-Decoder]  parameters   : {count_parameters(seq2seq):,}")
print()

[Encoder-Decoder]  output shape : torch.Size([4, 18, 32000])
[Encoder-Decoder]  parameters   : 93,287,680



In [13]:
# ── Encoder-Only / BERT-style ─────────────────────────────────────────────────
bert_model = BERTStyleEncoder(
    vocab_size=30_522,   # BERT's vocab size
    num_classes=2,       # binary classification (e.g. sentiment)
    d_model=768,
    num_heads=12,
    num_layers=6,        # BERT-base uses 12; using 6 here to keep smoke test fast
    d_ff=3072,
).to(DEVICE)

tokens = torch.randint(1, 30_522, (batch, 64)).to(DEVICE)
cls_logits = bert_model(tokens)
print(f"[Encoder-Only]     output shape : {cls_logits.shape}")
print(f"[Encoder-Only]     parameters   : {count_parameters(bert_model):,}")
print()

[Encoder-Only]     output shape : torch.Size([4, 2])
[Encoder-Only]     parameters   : 66,543,362



In [14]:
# ── Decoder-Only / GPT-style ──────────────────────────────────────────────────
gpt_model = GPTStyleDecoder(
    vocab_size=50_257,   # GPT-2's vocab size
    d_model=768,
    num_heads=12,
    num_layers=6,        # GPT-2 small uses 12; using 6 for speed
    d_ff=3072,
    max_seq_len=1024,
).to(DEVICE)

input_ids = torch.randint(1, 50_257, (batch, 32)).to(DEVICE)
lm_logits = gpt_model(input_ids)
print(f"[Decoder-Only]     output shape : {lm_logits.shape}")
print(f"[Decoder-Only]     parameters   : {count_parameters(gpt_model):,}")

[Decoder-Only]     output shape : torch.Size([4, 32, 50257])
[Decoder-Only]     parameters   : 81,107,712


---
## 12. Greedy Autoregressive Generation (GPT-style)

Demonstrates how the decoder-only model generates tokens one at a time, feeding each output back as input.

In [15]:
@torch.no_grad()
def greedy_generate(
    model: GPTStyleDecoder,
    prompt_ids: torch.Tensor,   # (1, prompt_len)
    max_new_tokens: int = 20,
    eos_token_id: int | None = None,
) -> torch.Tensor:
    model.eval()
    generated = prompt_ids.clone()

    for _ in range(max_new_tokens):
        logits = model(generated)                 # (1, cur_len, vocab)
        next_token_logits = logits[:, -1, :]      # (1, vocab) — last position
        next_token = next_token_logits.argmax(dim=-1, keepdim=True)  # (1, 1)
        generated = torch.cat([generated, next_token], dim=1)

        if eos_token_id is not None and next_token.item() == eos_token_id:
            break

    return generated


prompt = torch.randint(1, 50_257, (1, 5)).to(DEVICE)   # 5-token prompt
output = greedy_generate(gpt_model, prompt, max_new_tokens=10)
print(f"Prompt length    : {prompt.size(1)}")
print(f"Generated length : {output.size(1)}")
print(f"Token ids        : {output[0].tolist()}")

Prompt length    : 5
Generated length : 15
Token ids        : [347, 28733, 24444, 49262, 40861, 40861, 40861, 40861, 40861, 40861, 40861, 40861, 40861, 40861, 40861]
